# Phutball per-puzzle curriculum (v3; expert iteration, Qwen3.5-9B thinking on, board + men list)

Per-puzzle version of curriculum_colab.ipynb (llm_train/curriculum_pp.py). v2 (rounds 1-9) allocated by bucket averages:
with 300 of 15,512 puzzles a round no puzzle came up twice, so parole never fired, and cheap stable-cut "NO WIN"
credit starved the no-win bucket. Here every puzzle has its own success rate from **8 samples**, counting only answers
the model **finished on its own**:
- **round 1 = screen**: 1,600 puzzles stratified over the 11 buckets, 8 samples each (also round 1's training data);
- **8/8 -> mastered**: never sampled again, and no new training examples (their banked examples are replayed so the
  from-base retrain does not forget them); **0/8 -> paroled**; spot checks (10% of each round) bring paroled puzzles
  back when the model finishes one correctly;
- **later rounds**: 150 puzzles x 8 samples (same cost as v2's 300 x 4): active puzzles by priority (1-(1-p)^8)(1-p) of
  their own rate, no repeats in consecutive rounds, at most 30% from one bucket; fresh puzzles refill the working set,
  weighted toward buckets whose screened puzzles turned out learnable;
- **training set**: each round's new examples + equal replay from the bank (each puzzle's latest verified examples),
  at most 1,000, fine-tuned from the base model (as before);
- **gate** on a **validation set** (llm_train/make_val.py: 100 puzzles from each of the 10 other pool buckets and 60 prevent puzzles, including the
  backward-trap wins, never trained on), 1 sample, **paired and regression-only**: the new model is compared item by item
  with the best's stored results and promoted unless it is clearly worse (z < -1.5) on finished accuracy, on forced
  accuracy within wins / no-win / placements, or says NO WIN on clearly more real wins. Small real gains cannot be
  confirmed per round on ~1,000 items, so v2's "must beat the best" gate stalled; the benchmark (1,000 items) is now
  the **test set** only (round 1, every 5th round, the start), never used to choose a model.

**Prevent, regenerated**: in the v1 prevent puzzles one rule (a man next to the ball on your own side, an escape
hatch) saves 98-99%. The v2 puzzles (llm_train/make_prevent_v2.py, split_prevent_v2.py) rule escape-hatch squares out:
151 strict + 49 relaxed from every expert-game and self-play position, mirrored to 400 (pairs never split): 200 test,
60 validation, 140 training (`pool_v4`). No simple rule solves more than 29% (random placement 2.3%). The test set is
`bench_v2.jsonl`: the benchmark's 800 non-prevent items unchanged plus the 200 new prevent puzzles.

Starts from the **self-stop warm start**, the same model v2 started from, so v2 and v3 compare cleanly (same start,
curves over cumulative samples). v2's best (r9) is evaluated on the validation and test sets as a reference point. **Runtime:** A100, High-RAM. **Cost:** start evals ~1.7 h (the v2 reference ~1 h of that; `REF = False` skips it); round 1 (the screen) ~5 h
(~35 units); later rounds ~1.4 h (~10 units; the validation gate is ~35 min of that). Resumable: re-run all cells after a disconnect.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, glob, subprocess, time, urllib.request
OLD = '/content/drive/MyDrive/phutball/curriculum'            # v2 run
OUT = '/content/drive/MyDrive/phutball/curriculum_pp'; os.makedirs(OUT, exist_ok=True)
INIT = '/content/drive/MyDrive/phutball/grpo_pilot/self_stop/adapter/final'   # the warm start: v2's starting model too
REF = True; REF_ADAPTER = f'{OLD}/r9/adapter/final'                             # v2's best, evaluated for comparison only
POOL = 'llm_train/puzzles/pool_v4.jsonl.gz';     # pool_v2 minus the one-rule prevent puzzles, plus 140 regenerated ones
STATE = f'{OUT}/state.json'; BANK = f'{OUT}/bank.jsonl'; FMT = 'ascii+men'
TEST = 'llm_bench/data/bench_v2.jsonl'          # the benchmark with the regenerated prevent puzzles
VAL = 'llm_train/puzzles/val_v3.jsonl'; VAL_IDS = 'llm_train/puzzles/val_v3_ids.json'   # llm_train.make_val (in the repo)
CAP = 6144; SCREEN = 1600; N = 150; K = 8; ROUNDS = 15   # round 1 ~4.5 h, then ~1 h per round; raise ROUNDS any time (resumes)
PG = 25; EK = 4; FULL_EVERY = 5                   # test (benchmark): 25 per group x 4 samples, on round 1 and every 5th
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /content
!test -d phutball-jax || git clone -b expert-dagger https://github.com/echoname6/phutball-jax.git
%cd /content/phutball-jax
!git pull -q && git log --oneline -1
!pip -q install -U "trl[vllm]" peft datasets 2>&1 | tail -3
!pip -q install flash-linear-attention
!pip -q uninstall -y torchao
!python -c "import trl, vllm, peft, transformers; print('trl', trl.__version__, 'vllm', vllm.__version__, 'peft', peft.__version__, 'transformers', transformers.__version__)"

In [ ]:
def serve(adapters: dict, max_len=CAP + 2048):
    !pkill -f "vllm serve" ; sleep 5
    extra = ['--enable-lora', '--max-lora-rank', '32', '--max-loras', str(max(1, len(adapters))), '--lora-modules',
             *[f'{k}={v}' for k, v in adapters.items()]] if adapters else []
    log = open('/content/vllm_serve.log', 'w')
    p = subprocess.Popen(['vllm', 'serve', 'Qwen/Qwen3.5-9B', '--max-model-len', str(max_len), '--gpu-memory-utilization',
                          '0.90', '--port', '8000', *extra], stdout=log, stderr=subprocess.STDOUT)
    for _ in range(180):
        if p.poll() is not None: break
        try:
            urllib.request.urlopen('http://localhost:8000/health', timeout=5); print('server up:', list(adapters) or 'base'); return p
        except Exception: time.sleep(5)
    !tail -15 /content/vllm_serve.log
    raise RuntimeError('vLLM did not start')

EV = f'{OUT}/eval'; os.makedirs(EV, exist_ok=True)
def evaluate(name, model, samples=EK, bench=TEST, pg=PG):
    """bench: the test set (default) or VAL (gate: all 1,100 items, pg=0)."""
    if os.path.exists(f'{EV}/{name}-forced.summary.json'): print('done:', name); return
    !python -m llm_bench.run --bench {bench} --model {model} --name {name} --prompt-format {FMT} --thinking true --qwen-thinking-sampling --samples {samples} --max-tokens {CAP} --per-group {pg} --concurrency 96 | tail -14
    !python -m llm_bench.run --bench {bench} --model {model} --tokenizer Qwen/Qwen3.5-9B --prompt-format {FMT} --thinking true --salvage {name} --per-group {pg} --concurrency 48 | tail -14
    !cp llm_bench/results/{name}* {EV}/
def best_adapter():
    return (json.load(open(STATE)).get('best') if os.path.exists(STATE) else None) or INIT

## 1. Starting point

The warm start on the validation set (`v0`, the gate's first 'best') and on the test set (`start`, bench_v2: the prevent row differs from v2's baselines, so it is re-run); optionally v2's best (r9) on both, as a reference.

In [ ]:
todo = [('v0', 'warm', VAL, 0)] + ([('ref-v9', 'v2r9', VAL, 0), ('ref-r9', 'v2r9', TEST, PG)] if REF else [])
todo += [] if os.path.exists(f'{EV}/start-forced.summary.json') else [('start', 'warm', TEST, PG)]
todo = [t for t in todo if not os.path.exists(f'{EV}/{t[0]}-forced.summary.json')]
if todo:
    server = serve({'warm': INIT, **({'v2r9': REF_ADAPTER} if REF else {})})
    for name, model, bench, pg in todo:
        evaluate(name, model, samples=1 if bench == VAL else EK, bench=bench, pg=pg)
    !pkill -f "vllm serve"
else: print('start evals done')


## 2. Curriculum rounds (unattended)

In [ ]:
for R in range(1, ROUNDS + 1):
    d = f'{OUT}/r{R}'; os.makedirs(d, exist_ok=True)
    if not os.path.exists(f'{d}/ids.json'):
        if R == 1:
            !python -m llm_train.curriculum_pp screen --state {STATE} --pool {POOL} --exclude {VAL_IDS} --n {SCREEN} --out {d}/ids.json | head -16
        else:
            !python -m llm_train.curriculum_pp allocate --state {STATE} --pool {POOL} --exclude {VAL_IDS} --n {N} --round {R} --out {d}/ids.json | head -40
    if not os.path.exists(f'{d}/outcomes.jsonl'):
        print(f'=== round {R}: sampling from', best_adapter())
        !python -m llm_train.expert_iter --arm iid --adapter {best_adapter()} --pool {POOL} --ids {d}/ids.json --out {d} --k {K} --max-think {CAP} --seed {R} --prompt-format {FMT} --max-neg-ratio 1.0 2>&1 | grep --line-buffered -v -E "it/s\]|Processed prompts|Adding requests|INFO|WARNING"
    if not os.path.exists(f'{d}/train.jsonl'):
        !python -m llm_train.curriculum_pp update --state {STATE} --round {R} --outcomes {d}/outcomes.jsonl --sft {d}/sft.jsonl --bank {BANK} --train-out {d}/train.jsonl | tail -40
    if not os.path.exists(f'{d}/adapter/final/adapter_model.safetensors'):
        !python -m llm_train.sft_stop --data {d}/train.jsonl --out {d}/adapter --max-neg-ratio 1.0 2>&1 | grep --line-buffered -E "examples|'loss'|saved|rror|Traceback"
    full = (R == 1 or R % FULL_EVERY == 0)
    if not os.path.exists(f'{EV}/v{R}-forced.summary.json') or (full and not os.path.exists(f'{EV}/r{R}-forced.summary.json')):
        server = serve({f'r{R}': f'{d}/adapter/final'})
        evaluate(f'v{R}', f'r{R}', samples=1, bench=VAL, pg=0)     # gate: validation set
        if full: evaluate(f'r{R}', f'r{R}')                         # test set (curves only)
        !pkill -f "vllm serve"
    if not os.path.exists(f'{d}/gate.txt'):
        !python -m llm_train.curriculum_pp gate --state {STATE} --round {R} --new {EV}/v{R} --adapter {d}/adapter/final --init-best {EV}/v0 --init-adapter {INIT} | tee {d}/gate.txt
print('curriculum done; best:', best_adapter())


## 3. Results

In [ ]:
names = ['start', 'ref-r9'] + [f'r{R}' for R in range(1, ROUNDS + 1)]   # test evals: warm start, v2's best, round 1 and every FULL_EVERY
keys = ('win: positive, 1-jump', 'win: positive, 2-jump', 'win: positive, 3-jump', 'win: positive, 4-jump',
        'win: near-miss negative', 'forced placement', 'block placement', 'prevent placement')
vkeys = ('win: positive, 1-jump', 'win: positive, 2-jump', 'win: positive, 3-jump', 'win: positive, 4-jump',
         'win: backward-trap, 2-jump', 'win: backward-trap, 3-jump', 'win: backward-trap, 4-jump', 'win: near-miss negative',
         'forced placement', 'block placement', 'prevent placement')
vnames = ['v0', 'ref-v9'] + [f'v{R}' for R in range(1, ROUNDS + 1)]
for label, suffix in (('VALIDATION, FINISHED', '.summary.json'), ('VALIDATION, BUDGET-FORCED', '-forced.summary.json')):
    S = {n: json.load(open(f'{EV}/{n}{suffix}')) for n in vnames if os.path.exists(f'{EV}/{n}{suffix}')}
    print(f"\n{label}: pass@1, 100 per bucket")
    print(f"{'':28s}" + ''.join(f"{n:>7s}" for n in S))
    for k in vkeys:
        print(f"{k:28s}" + ''.join(f"{S[n][k]['accuracy']:>7.0%}" if k in S[n] else f"{'-':>7s}" for n in S))
    print(f"{'cut off (wins)':28s}" + ''.join(f"{S[n].get('win (all)', {}).get('truncated', 0):>7.0%}" for n in S))
for label, suffix in (('FINISHED ON ITS OWN', '.summary.json'), ('BUDGET-FORCED', '-forced.summary.json')):
    S = {n: json.load(open(f'{EV}/{n}{suffix}')) for n in names if os.path.exists(f'{EV}/{n}{suffix}')}
    print(f"\nTEST {label}: pass@1 / pass@{EK}  (ascii+men, thinking on, sampled, {CAP} tokens)")
    print(f"{'':26s}" + ''.join(f"{n:>14s}" for n in S))
    for k in keys:
        print(f"{k:26s}" + ''.join(f"{S[n][k]['accuracy']:>8.0%} /{S[n][k].get(f'pass@{EK}', float('nan')):>4.0%}" if k in S[n] else f"{'-':>14s}" for n in S))
    print(f"{'cut off (wins)':26s}" + ''.join(f"{S[n].get('win (all)', {}).get('truncated', 0):>14.0%}" for n in S))
    print(f"{'mean tokens':26s}" + ''.join(f"{S[n].get('mean_completion_tokens', 0):>14.0f}" for n in S))
if os.path.exists(STATE):
    st = json.load(open(STATE))
    print('\nscheduler log:')
    for e in st['log']:
        if e['event'] == 'gate': print(f"  round {e['round']}: {'PROMOTED' if e['promoted'] else 'kept best'} ({e['reason']})")
        elif e['event'] == 'update': print(f"  round {e['round']}: finished rates {e['finished_rates']} | {e['moves']} | {e['status']}")
        elif e['event'] == 'train_set': print(f"  round {e['round']}: train {e['new']} new + {e['replay']} replay ({e['dropped_mastered']} mastered dropped)")

In [ ]:
# optional: stop billing when everything above has finished (queue this cell last)
# from google.colab import runtime; runtime.unassign()